Uploading the model to Google Colab

In [ ]:
from google.colab import files
import os

print("Apni train ki hui 'model.h5' file upload karein:")
uploaded = files.upload()

if 'model.h5' in uploaded:
    print("\n'model.h5' safaltapoorvak upload ho gayi hai!")
else:
    print("\nError: 'model.h5' file nahi mili. Kripya file ka naam check karein ya phir se upload karein.")

for fn in uploaded.keys():
  print(f"Uploaded file '{fn}'")

Uploading Audio File

In [ ]:
from google.colab import files

print("\nApni test audio file (jaise 'my_test_file.wav') upload karein:")
uploaded_audio = files.upload()

if not uploaded_audio:
    print("\nError: Koi audio file upload nahi hui.")
else:
    # Upload ki gayi file ka naam store karein
    # Yeh maan raha hai ki aapne ek baar mein ek hi file upload ki hai
    audio_filename = list(uploaded_audio.keys())[0]
    print(f"\nAudio file '{audio_filename}' safaltapoorvak upload ho gayi hai!")
    print(f"Is naam ko agle cell mein istemal karein.")

Testing model and Downloading the output file

In [ ]:
# --- Step 1: Install libraries ---
!pip install noisereduce
!pip install soundfile  # <-- Audio save karne ke liye zaroori

# --- Step 2: Import everything ---
import pandas as pd
import numpy as np
import os
import librosa
from numpy import genfromtxt
from tensorflow.keras.models import load_model
import noisereduce as nr

import soundfile as sf
import IPython.display as ipd
import copy
from google.colab import files

# --- Step 3: Noise folder check ---
noise_dir = 'noise'
if not os.path.exists(noise_dir):
    os.makedirs(noise_dir)
print(f"Directory '{noise_dir}' maujood hai.")
print(f"--- ZAROORI ---")
print(f"Sunishchit karein ki '{noise_dir}' folder ke andar aapki saari noise files (ac1.wav, siren1.wav, etc.) uploaded hain.")


# --- FIXED DENOISE FUNCTION ---
def denoise(data_clip, noise_file_path, sr):
    print(f"Reducing noise using profile: {noise_file_path}")
    try:
        noise_clip, sr_noise = librosa.load(noise_file_path)
        # Perform noise reduction
        reduced_noise = nr.reduce_noise(audio_clip=data_clip,
                                        noise_clip=noise_clip,
                                        verbose=False)
        return reduced_noise
    except Exception as e:
        print(f"Error loading or processing noise file {noise_file_path}: {e}")
        return data_clip

# --- END FIX ---

# --- Step 4: Load Model ---
model_path = 'model.h5'
if not os.path.exists(model_path):
    print(f"Error: '{model_path}' nahi mili. Kripya Cell 1 phir se run karein.")
else:
    model = load_model(model_path)
    print('\n\n\n Model Loaded \n\n\n')

# --- Step 5: Set filename (Yeh automatically Cell 2 se aana chahiye) ---
try:
    filename = audio_filename
    print(f"Testing file: {filename}")
except NameError:
    print("Error: 'audio_filename' variable nahi mila.")
    filename = 'project explanation.mp3' # <-- Aapki file ka naam
    print(f"Fallback: Testing file: {filename}")
except Exception as e:
    print(f"File set karne mein error: {e}")


# --- Step 6: Preprocessing (FIXED) ---
print(f"Loading and preprocessing {filename}...")
x_test = []
try:
    y, sr = librosa.load(filename, sr=None)

    # 1. Extract MFCCs (Feature 1)
    mfccs = np.mean(librosa.feature.mfcc(y=y, sr=sr, n_mfcc=40).T, axis=0)
    # 2. Extract Melspectrogram (Feature 2)
    melspectrogram = np.mean(librosa.feature.melspectrogram(y=y, sr=sr, n_mels=40, fmax=8000).T, axis=0)

    # Stack karke (40, 2) shape banayein
    features = np.reshape(np.vstack((mfccs, melspectrogram)).T, (40, 2))
    # --- END FIX ---

    x_test.append(features)
    x_test = np.array(x_test)

    # Reshape for CNN input: (batch_size, 40, 2, 1)
    x_test = np.reshape(x_test, (x_test.shape[0], 40, 2, 1))
    print('\nFinal test shape: ', x_test.shape)

    # --- Step 7: Prediction ---
    ans = model.predict(x_test)

    print('Class 0: Windy \n Class 1: Horn\n Class 2: Children-noise \n Class 3: Dog Bark \n Class 4: Drilling \n Class 5: Engine Idling\n Class 6: Gun Shot \n Class 7: Jackhammer\n Class 8: Siren \n Class 9: Street music\n')

    my_dict = {0: 'Windy', 1: 'Horn', 2: 'Children-noise', 3: 'Dog Bark', 4: 'Drilling', 5: 'Engine Idling', 6: 'Gun Shot', 7: 'Jackhammer', 8: 'Siren', 9: 'Street music'}

    # Prediction logic
    x = copy.copy(ans[0])
    x = list(x)
    arr = []
    ls = list(ans[0])

    while (len(x) > 8): # Top 2 predictions
        aud = max(x)
        index = ls.index(aud)
        x.remove(aud)
        arr.append(index)

    print('Resulted Index: ', arr)
    print('\nNoises Present: ')
    print('')
    for idx in arr:
        print(my_dict[idx])

    # --- Step 8: Denoising (FIXED LOOP) ---
    print("\nStarting noise reduction...")
    cleaned_data = copy.copy(y)
    noise_files_to_use = []

    # Loop 1: Saari zaroori noise files ikattha karein
    for i in arr:
        if i == 0:
            noise_files_to_use.append("noise/ac1.wav")
            noise_files_to_use.append("noise/ac2.wav")
        elif i == 1:
            noise_files_to_use.append("noise/horn1.wav")
            noise_files_to_use.append("noise/horn2.wav")
        elif i == 2:
            noise_files_to_use.append("noise/children1.wav")
            noise_files_to_use.append("noise/children2.wav")
        elif i == 3:
            noise_files_to_use.append("noise/bark1.wav")
            noise_files_to_use.append("noise/bark2.wav")
        elif i == 4:
            noise_files_to_use.append("noise/drill1.wav")
            noise_files_to_use.append("noise/drill2.wav")
        elif i == 5:
            noise_files_to_use.append("noise/engine1.wav")
            noise_files_to_use.append("noise/engine2.wav")
        elif i == 6:
            noise_files_to_use.append("noise/engine1.wav")
            noise_files_to_use.append("noise/drill2.wav")
        elif i == 7:
            noise_files_to_use.append("noise/jack1.wav")
            noise_files_to_use.append("noise/jack2.wav")
        elif i == 8:
            noise_files_to_use.append("noise/siren1.wav")
            noise_files_to_use.append("noise/siren2.wav")
        elif i == 9:
            noise_files_to_use.append("noise/street1.wav")
            noise_files_to_use.append("noise/street2.wav")
        else:
            print(f'Index {i} ke liye noise profile defined nahi hai, skip kar raha hoon.')

    # Loop 2: Ek ke baad ek noise reduction apply karein
    for noise_file in noise_files_to_use:
        if os.path.exists(noise_file):
            cleaned_data = denoise(cleaned_data, noise_file, sr)
        else:
            print(f"Warning: Noise file {noise_file} nahi mili. Skip kar raha hoon.")

    # --- Step 9: Final result save aur play karein ---

    output_filename = 'clean.wav'

    # --- ERROR FIX: `soundfile.write` (sf.write) ka istemal kiya gaya ---
    sf.write(output_filename, cleaned_data, sr)
    print(f"\nSab noise reduction poora. Cleaned Audio save ho gaya hai: {output_filename}")

    # --- "HEAR" BUTTON ---
    print("\nOriginal Audio:")
    ipd.display(ipd.Audio(y, rate=sr))
    print("\nCleaned Audio:")
    ipd.display(ipd.Audio(cleaned_data, rate=sr))

    # --- "DOWNLOAD" BUTTON ---
    print(f"\n'{output_filename}' ko download karne ke liye button generate kar raha hoon...")
    files.download(output_filename)


except FileNotFoundError:
    print(f"Error: File '{filename}' nahi mili. Kya aapne use Cell 2 mein upload kiya hai?")
except Exception as e:
    print(f"Ek error aaya: {e}")